# 🤖 Agentic Basic RAG
### Using **Groq API** (LLM) + **HuggingFace Embeddings** (FAISS vector store)

**Pipeline Overview:**
```
User Query
    ↓
Agent decides: needs retrieval? → Retrieve from FAISS → Augment prompt → Groq LLM → Answer
                can answer directly? → Groq LLM → Answer
```

**Components:**
- 🔍 **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2` (free, no API key)
- 🗄️ **Vector Store**: FAISS (in-memory)
- 🧠 **LLM**: Groq (`llama3-8b-8192`) — ultra-fast inference
- 🕵️ **Agent**: decides when to retrieve vs answer directly

## Step 1 — Install Dependencies

In [ ]:
!pip install -q groq faiss-cpu sentence-transformers langchain langchain-community langchain-groq tiktoken

## Step 2 — Set API Keys

In [ ]:
import os
from getpass import getpass

# 🔑 Get your FREE Groq API key from: https://console.groq.com
GROQ_API_KEY = getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ API key set!")

## Step 3 — Import Libraries

In [ ]:
import os
import json
from typing import List, Dict, Any

# Groq
from groq import Groq

# HuggingFace Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# FAISS Vector Store
from langchain_community.vectorstores import FAISS

# Text Splitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Document loader helpers
from langchain.schema import Document

print("✅ All libraries imported!")

## Step 4 — Load & Prepare Documents

We'll use sample text documents. You can replace these with your own PDFs, URLs, etc.

In [ ]:
# =============================================
# 📄 Sample Documents — Replace with your own!
# =============================================
raw_documents = [
    """
    Groq is an AI infrastructure company that builds LPU (Language Processing Unit) chips
    specifically designed for AI inference. Their hardware allows extremely fast token generation,
    often 10x faster than traditional GPU-based inference. Groq's API provides access to
    open-source models like LLaMA 3, Mixtral, and Gemma at very high speeds.
    The Groq cloud service offers a free tier with generous rate limits, making it ideal
    for developers building AI applications.
    """,
    """
    RAG stands for Retrieval-Augmented Generation. It is a technique that combines
    information retrieval with language model generation. Instead of relying solely on
    the LLM's parametric knowledge, RAG first retrieves relevant documents from a
    knowledge base using semantic search (embeddings + vector store), then passes
    those documents as context to the LLM to generate a grounded, accurate answer.
    RAG reduces hallucinations and allows the model to answer questions about
    custom or recent data not in its training set.
    """,
    """
    HuggingFace is an AI company and open-source platform that hosts thousands of
    pre-trained machine learning models, datasets, and spaces. The `sentence-transformers`
    library from HuggingFace provides models specifically trained to generate meaningful
    sentence embeddings. The model `all-MiniLM-L6-v2` is a popular, lightweight embedding
    model that maps sentences to a 384-dimensional dense vector space and is widely used
    for semantic similarity, clustering, and retrieval tasks.
    """,
    """
    FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta AI
    for efficient similarity search and clustering of dense vectors. It is highly optimized
    for speed and memory usage and supports both CPU and GPU operations. In RAG pipelines,
    FAISS acts as the vector database — storing document embeddings and enabling fast
    nearest-neighbor search to find the most relevant chunks for a given query.
    """,
    """
    Agentic AI refers to AI systems that can autonomously plan, decide, and take actions
    to achieve goals. In agentic RAG, the LLM acts as an agent that decides whether to
    retrieve information from the knowledge base or answer directly based on its
    parametric knowledge. The agent can also decide to reformulate queries, retrieve
    multiple times, or skip retrieval for simple factual or conversational questions.
    This makes agentic RAG more flexible and efficient than naive RAG pipelines.
    """
]

# Wrap as LangChain Document objects
documents = [Document(page_content=text.strip(), metadata={"source": f"doc_{i}"}) 
             for i, text in enumerate(raw_documents)]

print(f"✅ Loaded {len(documents)} documents")

## Step 5 — Chunk the Documents

In [ ]:
# Split documents into smaller chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # characters per chunk
    chunk_overlap=50,      # overlap to preserve context at boundaries
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"\n📄 Sample chunk:")
print(chunks[0].page_content)

## Step 6 — Create HuggingFace Embeddings + FAISS Vector Store

In [ ]:
print("⏳ Loading HuggingFace embedding model (downloads on first run)...")

# Load free HuggingFace embedding model — no API key needed!
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},          # change to 'cuda' if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded!")
print("\n⏳ Building FAISS vector store...")

# Create FAISS vector store from chunks
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Create retriever — fetch top-3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"✅ FAISS vector store ready with {vectorstore.index.ntotal} vectors!")

## Step 7 — Initialize Groq Client

In [ ]:
# Initialize Groq client
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "llama3-8b-8192"   # Fast and capable. Alternatives: mixtral-8x7b-32768, llama3-70b-8192

def call_groq(messages: List[Dict], temperature: float = 0.2) -> str:
    """Helper to call Groq LLM."""
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=1024
    )
    return response.choices[0].message.content.strip()

print(f"✅ Groq client ready! Using model: {MODEL}")

## Step 8 — Build the Agentic RAG System

The **Agent** has two tools:
1. `retrieve_and_answer` — fetches relevant docs from FAISS, then generates a grounded answer
2. `direct_answer` — answers directly without retrieval (for simple/conversational queries)

The agent first decides which tool to use, then executes it.

In [ ]:
# ============================================================
# TOOL 1: Retrieve relevant docs and generate grounded answer
# ============================================================
def retrieve_and_answer(query: str) -> Dict[str, Any]:
    """Retrieve relevant docs from FAISS and answer using Groq."""
    
    # Step 1: Retrieve top-k relevant chunks
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Step 2: Build RAG prompt
    rag_prompt = f"""You are a helpful AI assistant. Use ONLY the following context to answer the user's question.
If the context doesn't contain enough information, say so honestly.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant that answers based on provided context."},
        {"role": "user", "content": rag_prompt}
    ]
    
    answer = call_groq(messages)
    
    return {
        "tool_used": "retrieve_and_answer",
        "retrieved_chunks": len(retrieved_docs),
        "sources": [doc.metadata.get("source", "unknown") for doc in retrieved_docs],
        "context_preview": context[:300] + "..." if len(context) > 300 else context,
        "answer": answer
    }


# ============================================================
# TOOL 2: Answer directly (no retrieval)
# ============================================================
def direct_answer(query: str) -> Dict[str, Any]:
    """Answer the query directly using Groq's parametric knowledge."""
    
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Answer the user's question concisely."},
        {"role": "user", "content": query}
    ]
    
    answer = call_groq(messages)
    
    return {
        "tool_used": "direct_answer",
        "retrieved_chunks": 0,
        "sources": [],
        "answer": answer
    }


# ============================================================
# AGENT ROUTER: Decides which tool to call
# ============================================================
def agent_decide(query: str) -> str:
    """Agent decides whether to retrieve or answer directly."""
    
    decision_prompt = f"""You are an AI agent that decides how to answer user queries.

You have access to a knowledge base containing information about:
- Groq (LPU chips, API, models)
- RAG (Retrieval-Augmented Generation)
- HuggingFace (embeddings, sentence-transformers)
- FAISS (vector store, similarity search)
- Agentic AI systems

For the user's query, decide:
- Use "retrieve_and_answer" if the query is about topics in the knowledge base or needs factual grounding
- Use "direct_answer" if it's a simple/generic/conversational question (greetings, math, general knowledge)

Respond with ONLY one of these two words: retrieve_and_answer OR direct_answer

User query: {query}
Decision:"""

    messages = [
        {"role": "user", "content": decision_prompt}
    ]
    
    decision = call_groq(messages, temperature=0.0).strip().lower()
    
    # Safety fallback
    if "retrieve" in decision:
        return "retrieve_and_answer"
    return "direct_answer"


# ============================================================
# MAIN AGENT FUNCTION
# ============================================================
def agentic_rag(query: str, verbose: bool = True) -> str:
    """
    Agentic RAG pipeline:
    1. Agent decides: retrieve or direct?
    2. Execute the chosen tool
    3. Return the answer
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🔍 Query: {query}")
        print(f"{'='*60}")
    
    # Step 1: Agent decides
    decision = agent_decide(query)
    
    if verbose:
        emoji = "📚" if decision == "retrieve_and_answer" else "💬"
        print(f"{emoji} Agent Decision: {decision}")
    
    # Step 2: Execute chosen tool
    if decision == "retrieve_and_answer":
        result = retrieve_and_answer(query)
        if verbose:
            print(f"📄 Retrieved {result['retrieved_chunks']} chunks from: {result['sources']}")
    else:
        result = direct_answer(query)
    
    if verbose:
        print(f"\n🤖 Answer:")
        print(result["answer"])
    
    return result["answer"]


print("✅ Agentic RAG system ready!")

## Step 9 — Test the Agentic RAG System

Let's test with queries that should trigger **retrieval** and some that should be answered **directly**.

In [ ]:
# ✅ Should trigger RETRIEVAL (topics in knowledge base)
_ = agentic_rag("What is RAG and how does it reduce hallucinations?")

In [ ]:
# ✅ Should trigger RETRIEVAL
_ = agentic_rag("What is Groq's LPU and why is it fast?")

In [ ]:
# ✅ Should trigger RETRIEVAL  
_ = agentic_rag("What embedding model does HuggingFace all-MiniLM-L6-v2 produce?")

In [ ]:
# 💬 Should be answered DIRECTLY (simple math / generic question)
_ = agentic_rag("What is 2 + 2?")

In [ ]:
# 💬 Should be answered DIRECTLY (greeting)
_ = agentic_rag("Hello! How are you?")

## Step 10 — Interactive Chat Loop

Run this cell to chat interactively with your Agentic RAG system!

In [ ]:
print("🤖 Agentic RAG Chat (type 'quit' to exit)")
print("-" * 50)

while True:
    user_input = input("\nYou: ").strip()
    
    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "q"]:
        print("👋 Bye!")
        break
    
    agentic_rag(user_input, verbose=True)

## ⚡ Bonus — Add Your Own Documents

Replace the sample docs with your own content!

In [ ]:
# =============================================
# Option A: Add plain text documents
# =============================================
def add_text_to_vectorstore(texts: List[str], source_name: str = "custom"):
    """Add new text documents to the existing vector store."""
    new_docs = [Document(page_content=t.strip(), metadata={"source": source_name}) for t in texts]
    new_chunks = text_splitter.split_documents(new_docs)
    vectorstore.add_documents(new_chunks)
    print(f"✅ Added {len(new_chunks)} new chunks to the vector store!")
    print(f"   Total vectors now: {vectorstore.index.ntotal}")

# Example usage:
# add_text_to_vectorstore(["""
#     Your custom document text here...
# """], source_name="my_docs")


# =============================================
# Option B: Load a PDF from Google Drive
# =============================================
# from langchain_community.document_loaders import PyPDFLoader
# from google.colab import drive
# drive.mount('/content/drive')
# 
# loader = PyPDFLoader("/content/drive/MyDrive/your_file.pdf")
# pdf_docs = loader.load()
# pdf_chunks = text_splitter.split_documents(pdf_docs)
# vectorstore.add_documents(pdf_chunks)
# print(f"Added {len(pdf_chunks)} chunks from PDF!")

print("✅ Bonus cell ready! Uncomment the examples to use them.")

## 📊 Architecture Summary

```
┌─────────────────────────────────────────────────┐
│                  USER QUERY                      │
└───────────────────────┬─────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────┐
│              AGENT ROUTER (Groq LLM)             │
│   Decides: retrieve_and_answer OR direct_answer  │
└───────┬─────────────────────────────┬────────────┘
        │ Knowledge query             │ Simple query
        ▼                             ▼
┌───────────────┐             ┌───────────────────┐
│  FAISS Retrieval│            │  Groq Direct LLM  │
│  (HF Embeddings)│            │  (No retrieval)   │
└───────┬────────┘             └────────┬──────────┘
        │ top-k chunks                  │
        ▼                               │
┌───────────────────┐                   │
│  Groq RAG Answer  │                   │
│  (context + query)│                   │
└───────┬───────────┘                   │
        └──────────────┬────────────────┘
                       ▼
                  FINAL ANSWER
```

| Component | Choice | Why |
|-----------|--------|-----|
| LLM | `llama3-8b-8192` via Groq | Ultra-fast, free tier |
| Embeddings | `all-MiniLM-L6-v2` | Free, no API key, 384-dim |
| Vector DB | FAISS | In-memory, fast, easy setup |
| Agent | Groq LLM router | Decides retrieve vs direct |